# Productivity Trends Analysis Data Preparation Notebook

This Notebook prepares the data which required for making impact analysis and predictive modeling.

In [59]:
# Disable warnings
import warnings
warnings.filterwarnings("ignore")

In [60]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
plt.style.use('ggplot')
sns.set_theme(style="whitegrid")

## 1. Load and Prepare Data
### Step 1: Extract Git Repository Data

The analysis uses two primary data sources extracted from Git repositories:

#### A. Commit Metadata

```bash
git log --date=iso-strict --pretty=format:'%H##%ad##%an##%ae##%s' > data/git_commits.csv
```

**Output Schema:**
- `sha`: Commit hash (unique identifier)
- `date`: ISO-8601 timestamp
- `author_name`: Contributor name
- `author_email`: Contributor email
- `subject`: Commit message

#### B. Code Change Statistics (numstat)

```bash
git log --numstat --date=iso-strict --pretty=format:'---%n%H##%ad##%an##%ae' > data/git_numstat.csv
```

**Output Schema:**
- Commit metadata (same as above)
- Per-file statistics:
  - `additions`: Lines added
  - `deletions`: Lines deleted
  - `path`: File path

In [61]:
# Load commit data
git_commit_pd = pd.read_csv("data/git_commits.csv", sep="##", header=None,
                      names=["sha","date","author_name","author_email","subject"],
                      engine="python")

# Date type data standardization
git_commit_pd["date"] = pd.to_datetime(git_commit_pd["date"], utc=True)
git_commit_pd["year"] = git_commit_pd["date"].dt.year
git_commit_pd["month"] = git_commit_pd["date"].dt.month
git_commit_pd["week"] = git_commit_pd["date"].dt.to_period("W").dt.start_time

git_commit_pd.head()

,sha,date,author_name,author_email,subject,year,month,week
0,942fbf30320a52f2f28e46961176e13cf8576a4d,2025-09-01 09:49:31+00:00,Brian Clozel,brian.clozel@broadcom.com,Polishing contribution,2025,9,2025-09-01
1,7b3c4e589301d45bf8f7b731192d2423671e71f4,2025-03-29 15:05:58+00:00,Mengqi Xu,2663479778@qq.com,Add support for Forwarded By HTTP headers,2025,3,2025-03-24
2,1653ec3b449ac81360dae9f916cc9aa18badea4c,2025-08-31 15:06:56+00:00,Park Sung Jun,junstin119@gmail.com,Add tests for applyRelativePath method in Stri...,2025,8,2025-08-25
3,746fc335c29a1dc0796cef5c790bf85b5c548463,2025-08-29 15:25:56+00:00,Sam Brannen,104798+sbrannen@users.noreply.github.com,Merge branch '6.2.x',2025,8,2025-08-25
4,b741632e99666f0469d64a7bf6dabd19368e709a,2025-08-29 15:25:15+00:00,Sam Brannen,104798+sbrannen@users.noreply.github.com,Polish wording in web sections,2025,8,2025-08-25


In [62]:
# Load numstat data (code changes)
rows = []
with open("data/git_numstat.csv", "r") as f:
    sha, date, author_name, author_email = None, None, None, None
    for line in f:
        line=line.rstrip("\n")
        if line == "---":
            sha = date = author_name = author_email = None
            continue
        if "##" in line and sha is None:
            sha, date, author_name, author_email = line.split("##", 3)
            date = pd.to_datetime(date, utc=True)
            continue
        if line and sha is not None:
            parts = line.split("\t")
            if len(parts) == 3:
                adds, dels, path = parts
                if adds != "-" and dels != "-":  # skip binary
                    rows.append((sha, date, author_name, author_email, int(adds), int(dels), path))

churn = pd.DataFrame(rows, columns=["sha","date","author_name", "author_email","adds","dels","path"])
churn["year"] = churn["date"].dt.year
churn["month"] = churn["date"].dt.month
churn["week"] = churn["date"].dt.to_period("W").dt.start_time

churn.head()

,sha,date,author_name,author_email,adds,dels,path,year,month,week
0,942fbf30320a52f2f28e46961176e13cf8576a4d,2025-09-01 09:49:31+00:00,Brian Clozel,brian.clozel@broadcom.com,0,4,framework-docs/modules/ROOT/pages/web/webflux/...,2025,9,2025-09-01
1,942fbf30320a52f2f28e46961176e13cf8576a4d,2025-09-01 09:49:31+00:00,Brian Clozel,brian.clozel@broadcom.com,10,1,framework-docs/modules/ROOT/partials/web/forwa...,2025,9,2025-09-01
2,942fbf30320a52f2f28e46961176e13cf8576a4d,2025-09-01 09:49:31+00:00,Brian Clozel,brian.clozel@broadcom.com,1,1,spring-web/src/main/java/org/springframework/h...,2025,9,2025-09-01
3,942fbf30320a52f2f28e46961176e13cf8576a4d,2025-09-01 09:49:31+00:00,Brian Clozel,brian.clozel@broadcom.com,0,1,spring-web/src/main/java/org/springframework/w...,2025,9,2025-09-01
4,942fbf30320a52f2f28e46961176e13cf8576a4d,2025-09-01 09:49:31+00:00,Brian Clozel,brian.clozel@broadcom.com,0,1,spring-web/src/main/java/org/springframework/w...,2025,9,2025-09-01


## 2. Calculate Contributor Experience

In [63]:
# Calculate the first and last contribution dates for each author
author_experience = churn.groupby("author_name").agg(
    first_contribution=pd.NamedAgg(column="date", aggfunc="min"),
    last_contribution=pd.NamedAgg(column="date", aggfunc="max"),
    total_commits=pd.NamedAgg(column="sha", aggfunc="nunique"),
    total_files_changed=pd.NamedAgg(column="path", aggfunc="count"),
    total_additions=pd.NamedAgg(column="adds", aggfunc="sum"),
    total_deletions=pd.NamedAgg(column="dels", aggfunc="sum")
)

# Calculate experience in years
current_date = pd.Timestamp.now(tz='UTC')
author_experience["years_since_first_commit"] = (current_date - author_experience["first_contribution"]).dt.days / 365.25
author_experience["active_years"] = (author_experience["last_contribution"] - author_experience["first_contribution"]).dt.days / 365.25
author_experience["total_changes"] = author_experience["total_additions"] + author_experience["total_deletions"]

# Sort by years of experience
author_experience_sorted = author_experience.sort_values(by="years_since_first_commit", ascending=False)

# Display top 20 most experienced contributors
author_experience_sorted.head(20)

,first_contribution,last_contribution,total_commits,total_files_changed,total_additions,total_deletions,years_since_first_commit,active_years,total_changes
author_name,,,,,,,,,
Ben Hale,2008-07-11 06:34:50+00:00,2010-08-23 13:17:31+00:00,31,120,2049,528,17.240246,2.116359,2577
Andy Clement,2008-08-11 18:37:11+00:00,2020-09-05 05:19:17+00:00,259,1474,121776,81377,17.155373,12.065708,203153
Arjen Poutsma,2008-10-21 08:04:24+00:00,2024-07-01 14:26:49+00:00,1565,11900,762326,100254,16.960986,15.693361,862580
Scott Andrews,2008-10-24 21:05:25+00:00,2012-05-31 17:45:10+00:00,39,190,3761,489,16.952772,3.597536,4250
Juergen Hoeller,2008-10-24 10:03:04+00:00,2025-08-24 08:31:01+00:00,7738,37895,545680,599552,16.952772,16.829569,1145232
Thomas Risberg,2008-11-07 20:21:08+00:00,2017-01-06 21:34:45+00:00,157,489,113985,29904,16.914442,8.164271,143889
Chris Beams,2008-11-24 22:16:21+00:00,2014-11-13 10:02:50+00:00,911,21891,270734,202603,16.867899,5.965777,473337
Costin Leau,2008-11-25 22:20:53+00:00,2012-01-06 16:12:25+00:00,172,941,16517,12601,16.865161,3.110198,29118
Christian Dupuis,2008-11-29 04:46:36+00:00,2013-11-11 15:23:10+00:00,22,67,464,155,16.856947,4.950034,619


## 3. Calculate Contributor Impact

### Contribution Impact Score

Impact score is score calculated using coefficients assigned to various contribution metrics. The metric does not contain any unit. These coefficients are based on product development team that focuses on new features. Coefficients can be vary based on the team focus, either refactoring, bug fixing or new features.    

In [64]:
# Calculate impact score for each contributor
# We'll define impact as a weighted combination of:
# - Number of commits
# - Code changes (additions weighted more than deletions)
# - Files touched

author_experience["impact_score"] = (
    author_experience["total_commits"] * 5 +  # Each commit is worth 5 points
    author_experience["total_additions"] * 2 +  # Each line added is worth 2 points
    author_experience["total_deletions"] * 0.5 +  # Each line deleted is worth 0.5 points
    author_experience["total_files_changed"] * 0.2  # Each file touched is worth 0.2 points
)

# Sort by impact score
impact_sorted = author_experience.sort_values(by="impact_score", ascending=False)

# Display top 20 contributors by impact
impact_sorted.head(20)

,first_contribution,last_contribution,total_commits,total_files_changed,total_additions,total_deletions,years_since_first_commit,active_years,total_changes,impact_score
author_name,,,,,,,,,,
Arjen Poutsma,2008-10-21 08:04:24+00:00,2024-07-01 14:26:49+00:00,1565,11900,762326,100254,16.960986,15.693361,862580,1584984.0
Juergen Hoeller,2008-10-24 10:03:04+00:00,2025-08-24 08:31:01+00:00,7738,37895,545680,599552,16.952772,16.829569,1145232,1437405.0
Rossen Stoyanchev,2009-03-07 00:08:49+00:00,2024-10-15 16:13:29+00:00,3804,20276,476724,231816,16.588638,15.608487,708540,1092431.2
Sam Brannen,2009-04-27 22:49:34+00:00,2025-08-29 15:25:15+00:00,5692,27080,335078,229726,16.446270,16.336756,564804,818895.0
Chris Beams,2008-11-24 22:16:21+00:00,2014-11-13 10:02:50+00:00,911,21891,270734,202603,16.867899,5.965777,473337,651702.7
Phillip Webb,2012-06-22 00:32:33+00:00,2025-08-11 16:32:27+00:00,648,15256,230372,205098,13.295003,13.136208,435470,569584.2
Brian Clozel,2013-09-24 10:03:53+00:00,2025-09-01 09:49:31+00:00,1275,4673,169638,124190,12.035592,11.934292,293828,408680.6
Rob Winch,2012-07-10 00:11:08+00:00,2024-05-22 02:49:24+00:00,127,3359,142337,140394,13.245722,11.865845,282731,356177.8
Andy Clement,2008-08-11 18:37:11+00:00,2020-09-05 05:19:17+00:00,259,1474,121776,81377,17.155373,12.065708,203153,285830.3


## 4. Yearly Impact Analysis

Contributor impact over time is analyzed to see how contributions have evolved year by year. This helps identify consistent contributors and those who had significant impact in specific years.

In [65]:
# Calculate yearly impact for each contributor
yearly_impact = churn.groupby(["year", "author_name"]).agg(
    commits=pd.NamedAgg(column="sha", aggfunc="nunique"),
    files_changed=pd.NamedAgg(column="path", aggfunc="count"),
    additions=pd.NamedAgg(column="adds", aggfunc="sum"),
    deletions=pd.NamedAgg(column="dels", aggfunc="sum")
)

# Calculate impact score for each year
yearly_impact["impact_score"] = (
    yearly_impact["commits"] * 5 +
    yearly_impact["additions"] * 2 +
    yearly_impact["deletions"] * 0.5 +
    yearly_impact["files_changed"] * 0.2
)

# Reset index to make year and author_name regular columns
yearly_impact = yearly_impact.reset_index()

# Get top 10 contributors by total impact
top_contributors = impact_sorted.head(10).index.tolist()

# Filter yearly impact for top contributors
top_yearly_impact = yearly_impact[yearly_impact["author_name"].isin(top_contributors)]

# Display sample of yearly impact data
top_yearly_impact.head()

,year,author_name,commits,files_changed,additions,deletions,impact_score
0,2008,Andy Clement,105,478,83220,63738,198929.6
1,2008,Arjen Poutsma,174,5602,549905,20565,1112082.9
3,2008,Chris Beams,135,2733,50387,26576,115283.6
6,2008,Juergen Hoeller,37,890,11260,19841,32803.5
9,2009,Andy Clement,72,553,13172,13663,33646.1


## 5. Create a Comprehensive Contributor Dataset

This section compiles a comprehensive dataset that includes all relevant metrics for each contributor. Additional metrics such as average additions/deletions per commit, commits per year, recency score, consistency score, and a weighted impact score are calculated to provide a holistic view of each contributor's performance and engagement over time.

In [66]:
# Create a comprehensive dataset with all metrics
contributor_dataset = author_experience.copy()

# Add additional metrics
contributor_dataset["avg_additions_per_commit"] = contributor_dataset["total_additions"] / contributor_dataset["total_commits"]
contributor_dataset["avg_deletions_per_commit"] = contributor_dataset["total_deletions"] / contributor_dataset["total_commits"]
contributor_dataset["avg_files_per_commit"] = contributor_dataset["total_files_changed"] / contributor_dataset["total_commits"]
contributor_dataset["commits_per_year"] = contributor_dataset["total_commits"] / contributor_dataset["active_years"]

# Calculate recency score (higher for more recent contributions)
max_date = contributor_dataset["last_contribution"].max()
contributor_dataset["recency_score"] = 1 - ((max_date - contributor_dataset["last_contribution"]).dt.days / 365.25) / 10
contributor_dataset["recency_score"] = contributor_dataset["recency_score"].clip(0, 1)  # Clip between 0 and 1

# Calculate consistency score (ratio of active years to years since first commit)
contributor_dataset["consistency_score"] = contributor_dataset["active_years"] / contributor_dataset["years_since_first_commit"]
contributor_dataset["consistency_score"] = contributor_dataset["consistency_score"].clip(0, 1)  # Clip between 0 and 1

# Calculate weighted impact score that considers recency and consistency
contributor_dataset["weighted_impact_score"] = (
    contributor_dataset["impact_score"] * 
    (0.7 + 0.15 * contributor_dataset["recency_score"] + 0.15 * contributor_dataset["consistency_score"])
)

# Sort by weighted impact score
weighted_impact_sorted = contributor_dataset.sort_values(by="weighted_impact_score", ascending=False)

# Display top 20 contributors by weighted impact
weighted_impact_sorted[["years_since_first_commit", "active_years", "total_commits", 
                        "impact_score", "recency_score", "consistency_score", "weighted_impact_score"]].head(20)

,years_since_first_commit,active_years,total_commits,impact_score,recency_score,consistency_score,weighted_impact_score
author_name,,,,,,,
Arjen Poutsma,16.960986,15.693361,1565,1584984.0,0.883368,0.925262,1.539486e+06
Juergen Hoeller,16.952772,16.829569,7738,1437405.0,0.997810,0.992733,1.435366e+06
Rossen Stoyanchev,16.588638,15.608487,3804,1092431.2,0.912389,0.940914,1.068393e+06
Sam Brannen,16.446270,16.336756,5692,818895.0,0.999452,0.993341,8.180098e+05
Phillip Webb,13.295003,13.136208,648,569584.2,0.994524,0.988056,5.680959e+05
Chris Beams,16.867899,5.965777,911,651702.7,0.000000,0.353676,4.907657e+05
Brian Clozel,12.035592,11.934292,1275,408680.6,1.000000,0.991583,4.081646e+05
Rob Winch,13.245722,11.865845,127,356177.8,0.872142,0.895825,3.437811e+05
Andy Clement,17.155373,12.065708,259,285830.3,0.501164,0.703320,2.517229e+05


## 6. Save the Contributor Dataset

We will save this dataset for further analysis or reporting.

In [67]:
# Save the comprehensive contributor dataset to CSV
contributor_dataset.reset_index().to_csv("data/contributor_impact_dataset.csv", index=False)

print(f"Saved contributor dataset with {len(contributor_dataset)} contributors")

Saved contributor dataset with 1166 contributors
